# Python Excel Automation: รวม Sheet ใน Excel เป็น Sheet เดียวด้วย xlwings #6

**คำอธิบาย:** เรียนรู้วิธีรวม sheet ใน Excel เป็น sheet เดียวด้วย xlwings ถ้าคุณได้รับข้อมูลในหลาย sheet หรือหลาย workbook ที่ต้องการสรุปรวม วิธี Consolidate จะช่วยรวมข้อมูลทั้งหมดไว้ใน sheet เดียว

In [ ]:
# 📦 ติดตั้ง package ที่จำเป็น (ข้ามถ้าติดตั้งแล้ว)
try:
    import xlwings
    import pandas
    print("✅ ติดตั้ง package แล้ว")
except ImportError:
    %pip install xlwings pandas

✅ Packages already installed.


In [ ]:
# 📚 Import ไลบรารี
# xlwings — เปิด Excel workbook และคัดลอกข้อมูลระหว่าง sheet
# pandas — ใช้สำหรับวิธีรวมทางเลือก (มีประสิทธิภาพกว่า)
import xlwings as xw
import pandas as pd

## 1. เปิด Workbook ที่มีหลาย Sheet

In [ ]:
# 📂 เปิด workbook ที่มีหลาย sheet เพื่อรวม
# ⚠️ ต้องมีไฟล์ 'Split_Data_xlwings.xlsx' — รัน Notebook 07 ก่อนเพื่อสร้างไฟล์นี้!
file_name = "Split_Data_xlwings.xlsx"
wkb = xw.Book(file_name)

print(f"Workbook: {wkb.name}")
print(f"Sheet ทั้งหมด ({len(wkb.sheets)}):")
for s in wkb.sheets:
    print(f"  - {s.name}")

Workbook: Split_Data_xlwings.xlsx
Sheets (3):
  - Sheet1
  - Order
  - Filtered


## 2. สร้าง Sheet สำหรับรวมข้อมูล

In [ ]:
# ➕ สร้าง sheet ว่างใหม่เพื่อเก็บข้อมูลที่รวมทั้งหมด
# after=wkb.sheets[-1] วางไว้ท้ายสุดของ workbook
combined_sheet = wkb.sheets.add(name='Combined', after=wkb.sheets[-1])
print(f"สร้าง sheet 'Combined' แล้ว")

Created 'Combined' sheet


## 3. คัดลอกหัวข้อจาก Sheet แรก

In [ ]:
# 📋 คัดลอกแถวหัวข้อจาก sheet แรกไปยัง sheet Combined
# .current_region.rows[0] ดึงเฉพาะแถวแรก (หัวข้อ)
# เขียนลง sheet Combined เพื่อให้ข้อมูลทั้งหมดมีหัวข้อ
source_sheet = wkb.sheets[0]

header_range = source_sheet.range('A1').current_region.rows[0]
header_values = header_range.value
combined_sheet.range('A1').value = header_values
print(f"คัดลอกหัวข้อแล้ว: {header_values}")

Header copied: ['Sales Representative', 'Location', 'Region', 'Customer', 'Order Date', 'Product', 'Quantity', 'Price', 'Total Sale Amount']


## 4. รวมข้อมูลจากทุก Sheet

In [ ]:
# 🔗 รวมข้อมูลจากทุก sheet ด้วย xlwings (วิธีทีละเซลล์)
# ตรรกะ: วนลูปแต่ละ sheet, ข้าม 'Combined', อ่านข้อมูล (ไม่มีหัวข้อ), วางลง Combined
# current_row ติดตามว่าจะวางที่ไหนต่อไป — เริ่มที่แถว 2 (หลังหัวข้อ)
current_row = 2

for sheet in wkb.sheets:
    if sheet.name == 'Combined':
        continue  # ข้าม sheet ปลายทาง
    
    # ดึงช่วงข้อมูล (ไม่รวมหัวข้อ)
    data_range = sheet.range('A1').current_region
    if data_range.rows.count <= 1:
        continue  # ข้าม sheet ว่าง (มีแค่หัวข้อ)
    
    # อ่านข้อมูลที่ไม่มีแถวหัวข้อ (จาก A2 ถึงเซลล์สุดท้าย)
    data = sheet.range(
        f'A2:{data_range.columns[-1].get_address(False, False)[0]}{data_range.rows.count}'
    ).value
    
    if data:
        # จัดการกรณีแถวเดียว (xlwings คืนค่า flat list สำหรับ 1 แถว)
        if not isinstance(data[0], list):
            data = [data]
        
        # วางข้อมูลลง sheet Combined ที่แถวปัจจุบัน
        combined_sheet.range(f'A{current_row}').value = data
        rows_added = len(data)
        print(f"Sheet '{sheet.name}': คัดลอก {rows_added} แถว")
        current_row += rows_added

print(f"\n✅ จำนวนแถวที่รวมทั้งหมด: {current_row - 2}")

Sheet 'Sheet1': 5684 rows copied
Sheet 'Order': 9994 rows copied
Sheet 'Filtered': 3597 rows copied

✅ Total rows combined: 19275


In [ ]:
# 📐 ปรับความกว้างคอลัมน์อัตโนมัติ — ปรับให้พอดีกับเนื้อหา
combined_sheet.autofit(axis='columns')
print("✅ ปรับความกว้างคอลัมน์อัตโนมัติแล้ว")

✅ Columns auto-fitted


## 5. ทางเลือก: รวมข้อมูลด้วย Pandas (มีประสิทธิภาพกว่า)

In [ ]:
# 🐼 ทางเลือก: รวมด้วย Pandas (มีประสิทธิภาพกว่ามากสำหรับข้อมูลขนาดใหญ่!)
# pd.concat() ต่อหลาย DataFrame ในแนวตั้ง
# ignore_index=True รีเซ็ตหมายเลขแถว (0, 1, 2, ...)
all_dfs = []

for sheet in wkb.sheets:
    if sheet.name in ['Combined', 'Combined_Pandas']:
        continue
    
    # อ่านแต่ละ sheet เป็น DataFrame
    data = sheet.range('A1').current_region.options(pd.DataFrame, header=1).value
    if len(data) > 0:
        all_dfs.append(data)
        print(f"อ่าน {len(data)} แถวจาก '{sheet.name}'")

# ต่อ DataFrame ทั้งหมดเป็นอันเดียว
combined_df = pd.concat(all_dfs, ignore_index=True)
print(f"\n✅ รวมทั้งหมด: {len(combined_df)} แถว")
combined_df.head()

Read 5684 rows from 'Sheet1'
Read 9994 rows from 'Order'
Read 3597 rows from 'Filtered'

✅ Total combined: 19275 rows


,Location,Region,Customer,Order Date,Product,Quantity,Price,Total Sale Amount,Order ID,Ship Date,...,State,Postal Code,Product ID,Category,Sub-Category,Product Name,Sales,Discount,Profit,Sales Grade
0,Massachusetts,East,Raymond Young,2016-01-01 00:00:00,"Bravo II Megaboss 12-Amp Hard Body Upright, Re...",6.0,12.42,74.52,NaN,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,New York,East,Helen Dean,2016-01-01 00:00:00,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",7.0,12.42,86.94,NaN,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Washington,West,Shirley Chavez,2016-01-01 00:00:00,Acme Hot Forged Carbon Steel Scissors with Nic...,2.0,16.32,32.64,NaN,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,New Jersey,East,Brian Ryan,2016-01-01 00:00:00,Bretford CR4500 Series Slim Rectangular Table,1.0,12.42,12.42,NaN,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,New Jersey,East,Benjamin Willis,2016-01-01 00:00:00,Eldon Fold 'N Roll Cart System,3.0,17.83,53.49,NaN,NaT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# ✍️ เขียน DataFrame ที่รวมด้วย pandas ลง sheet ใหม่
# ตรวจสอบว่า sheet มีอยู่แล้วหรือไม่เพื่อหลีกเลี่ยงข้อผิดพลาดเมื่อรันซ้ำ
if 'Combined_Pandas' not in [s.name for s in wkb.sheets]:
    pandas_sheet = wkb.sheets.add(name='Combined_Pandas', after=wkb.sheets[-1])
else:
    pandas_sheet = wkb.sheets['Combined_Pandas']

pandas_sheet.range('A1').options(pd.DataFrame, index=False).value = combined_df
pandas_sheet.autofit(axis='columns')
print("✅ เขียนข้อมูลที่รวมด้วยวิธี Pandas แล้ว")

✅ Combined data written using Pandas method


## 6. บันทึก

In [ ]:
# 💾 บันทึก workbook พร้อมข้อมูลที่รวมแล้ว
wkb.save('Combined_Excel_Sheets.xlsx')
print("✅ บันทึก workbook เป็น 'Combined_Excel_Sheets.xlsx' แล้ว!")

# ปิดเมื่อต้องการ (uncomment)
# wkb.close()

✅ Workbook saved as 'Combined_Excel_Sheets.xlsx'!


---

## 🎮 Mini Project: รวม Sheet อัตโนมัติ

ทดสอบการรวมข้อมูลจากหลาย Sheet ด้วย xlwings!

> 💡 **คำตอบ:** ดูไฟล์ `answers/13_answers.ipynb`

---

### โจทย์ที่ 1: รวม Sheet + เพิ่มคอลัมน์ Source 🟢
ใช้ไฟล์ที่สร้างจาก NB 12 (`split_by_category_xlwings.xlsx`):
1. เปิดไฟล์ด้วย xlwings
2. อ่านข้อมูลจากทุก Sheet (ยกเว้น Summary)
3. เพิ่มคอลัมน์ `Source_Sheet` ในแต่ละ DataFrame
4. รวมด้วย `pd.concat()`
5. เขียนไปยัง sheet ใหม่ `"Combined"` + autofit
6. Print จำนวนแถวรวม

In [ ]:
# โจทย์ที่ 1: รวม Sheet + เพิ่มคอลัมน์ Source
# เขียนโค้ดของคุณที่นี่ 👇



### โจทย์ที่ 2: รวม + หา Top 5 ทุก Category 🟡
1. จากข้อมูลที่รวมแล้ว กรอง Top 5 ยอดขายสูงสุดของแต่ละ Category
2. เขียนผลลัพธ์ไปยัง sheet `"Top5_Each_Category"`
3. Autofit + Print สรุป

> 💡 Hint: `df.groupby(col).apply(lambda x: x.nlargest(5, 'Total Revenue'))`

In [ ]:
# โจทย์ที่ 2: รวม + หา Top 5 ทุก Category
# เขียนโค้ดของคุณที่นี่ 👇

